<a href="https://colab.research.google.com/github/nareshkumar01062007-rgb/SEARCH-TECHNIQUES/blob/main/memorybound.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import heapq

# Goal state
GOAL = (1, 2, 3,
        4, 5, 6,
        7, 8, 0)

# Manhattan Distance
def heuristic(state):
    distance = 0

    for i, value in enumerate(state):
        if value == 0:
            continue

        goal_index = GOAL.index(value)

        row1, col1 = divmod(i, 3)
        row2, col2 = divmod(goal_index, 3)

        distance += abs(row1 - row2) + abs(col1 - col2)

    return distance


# Generate possible moves
def get_neighbors(state):
    neighbors = []

    zero = state.index(0)
    row, col = divmod(zero, 3)

    moves = [
        (-1, 0),   # Up
        (1, 0),    # Down
        (0, -1),   # Left
        (0, 1)     # Right
    ]

    for dr, dc in moves:
        new_row = row + dr
        new_col = col + dc

        if 0 <= new_row < 3 and 0 <= new_col < 3:

            new_zero = new_row * 3 + new_col

            new_state = list(state)
            new_state[zero], new_state[new_zero] = \
                new_state[new_zero], new_state[zero]

            neighbors.append(tuple(new_state))

    return neighbors


# SMA* Search
def sma_star(start, memory_limit=50):

    # (f, counter, state, path, g)
    open_list = []

    counter = 0

    g = 0
    h = heuristic(start)
    f = g + h

    heapq.heappush(
        open_list,
        (f, counter, start, [start], g)
    )

    visited = {}

    while open_list:

        f, _, state, path, g = heapq.heappop(open_list)

        # Goal test
        if state == GOAL:
            return path

        # Avoid worse paths
        if state in visited and visited[state] <= g:
            continue

        visited[state] = g

        # Generate children
        children = []

        for next_state in get_neighbors(state):

            if next_state in path:
                continue

            new_g = g + 1
            h = heuristic(next_state)
            new_f = new_g + h

            counter += 1

            children.append(
                (new_f, counter, next_state,
                 path + [next_state], new_g)
            )

        # Add children
        for child in children:
            heapq.heappush(open_list, child)

        # Memory bounded condition
        while len(open_list) > memory_limit:

            # Remove worst node
            worst = max(
                open_list,
                key=lambda x: x[0]
            )

            open_list.remove(worst)
            heapq.heapify(open_list)

        counter += 1

    return None


# Display puzzle
def print_puzzle(state):
    for i in range(0, 9, 3):
        print(state[i:i+3])
    print()


# Example initial state
start = (
    1, 2, 3,
    4, 0, 6,
    7, 5, 8
)

print("Initial State:")
print_puzzle(start)

solution = sma_star(start, memory_limit=20)

if solution:
    print("Solution found!")
    print("Number of moves:", len(solution) - 1)

    for step, state in enumerate(solution):
        print("Step", step)
        print_puzzle(state)
else:
    print("No solution found.")